# Household Electric Power Consumption: Data Analysis and Visualization

**Author:** Suhaiv Man Singh &nbsp;|&nbsp; **Dataset:** UCI *Individual Household Electric Power Consumption* (January–June 2007 subset) &nbsp;|&nbsp; **Tools:** Python, Pandas, Matplotlib, Seaborn, scikit-learn

## Objective

The objective of this analysis is to understand household electricity consumption patterns over time and to identify the key factors that drive energy usage. Using one-minute electricity readings, the notebook cleans and transforms the data, explores it with statistics and visualisations, and draws insights about daily and weekly usage behaviour, peak demand, and the relationship between total consumption and the individual appliance groups.

**Scope.** The task covers data cleaning, transformation, exploratory data analysis (EDA), visualisation and insights.

## 0. Dataset source and background

**Source:** *Individual Household Electric Power Consumption*, UCI Machine Learning Repository (dataset id 235).

| Item | Detail |
|---|---|
| Creators | Georges Hebrail (Senior Researcher, EDF R&D, Clamart, France) and Alice Berard (TELECOM ParisTech, Master of Engineering internship at EDF R&D) |
| Original coverage | 2,075,259 one-minute measurements from a house in **Sceaux (7 km from Paris, France)**, December 2006 to November 2010 (47 months) |


**Suggested citation:** Hebrail, G. & Berard, A. (2006). *Individual Household Electric Power Consumption* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C58K54

**What this notebook uses.** The provided file `power.csv` is a six-month subset: 1 January to 30 June 2007 at one-minute resolution, which is 181 days × 1,440 minutes = **260,640 rows** (about 12.6% of the full archive). A subset of 6 months data is extracted from the orginal `txt` file and converted to `csv` format for our analysis. 

### Variables
| Variable | Unit | Meaning |
|---|---|---|
| `Date`, `Time` | dd-mm-yyyy, hh:mm:ss | Timestamp of the one-minute reading |
| `Global_active_power` | kW | Household minute-averaged **active** power (energy actually consumed) |
| `Global_reactive_power` | kVAR | Household minute-averaged **reactive** power (circulates but does no useful work) |
| `Voltage` | V | Minute-averaged voltage |
| `Global_intensity` | A | Household minute-averaged current intensity |
| `Sub_metering_1` | Wh | Energy sub-metering No. 1. It corresponds to the kitchen, containing mainly a dishwasher, an oven and a microwave (hot plates are not electric but gas powered). |
| `Sub_metering_2` | Wh | Energy sub-metering No. 2. It corresponds to the laundry room, containing a washing-machine, a tumble-drier, a refrigerator and a light. |
| `Sub_metering_3` | Wh | Energy sub-metering No. 3. It corresponds to an electric water-heater and an air-conditioner. |

## 1. Data collection and exploration

### 1.1 Dataset loading

The raw file is loaded, with no type conversion, to understand the dataset and figure out the issues in it.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.preprocessing import StandardScaler, MinMaxScaler  

df_power = pd.read_csv('Dataset/power.csv', index_col='index')

_counter = {'Table': 0, 'Fig': 0}

def caption(kind, text):
    """Display an auto-numbered caption such as 'Table 3. ...' or 'Fig 5. ...'."""
    _counter[kind] += 1
    display(Markdown(f'<div style="margin-left: 10px;"><strong>{kind} {_counter[kind]}.</strong> {text}</div>'))

print('Dataset loaded successfully!')
print(f'Shape: {df_power.shape[0]:,} rows × {df_power.shape[1]} columns')
display(df_power.head())
caption('Table', 'First five rows of the raw dataset')

Dataset loaded successfully!
Shape: 260,640 rows × 9 columns


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
index,,,,,,,,,
0,01-01-2007,00:00:00,2.58,0.136,241.97,10.6,0,0,0.0
1,01-01-2007,00:01:00,2.552,0.1,241.75,10.4,0,0,0.0
2,01-01-2007,00:02:00,2.55,0.1,241.64,10.4,0,0,0.0
3,01-01-2007,00:03:00,2.55,0.1,241.71,10.4,0,0,0.0
4,01-01-2007,00:04:00,2.554,0.1,241.98,10.4,0,0,0.0


<div style="margin-left: 10px;"><strong>Table 1.</strong> First five rows of the raw dataset</div>

### 1.2 Initial exploration: structure and data types

In [5]:
info_table = pd.DataFrame({
    'Column': df_power.columns,
    'Data Type': [str(t) for t in df_power.dtypes],
    'Non-null': df_power.notna().sum().values,
    'Distinct values': df_power.nunique().values,
})
display(info_table)
caption('Table', 'Structure of the raw dataset')

,Column,Data Type,Non-null,Distinct values
0,Date,object,260640,181
1,Time,object,260640,1440
2,Global_active_power,object,260640,3585
3,Global_reactive_power,object,260640,414
4,Voltage,object,260640,2276
5,Global_intensity,object,260640,200
6,Sub_metering_1,object,260640,79
7,Sub_metering_2,object,260640,80
8,Sub_metering_3,float64,256869,21


<div style="margin-left: 10px;"><strong>Table 2.</strong> Structure of the raw dataset</div>